## Full MLOps Pipeline for a Credit Scoring Model

Guide to deploying, monitoring, and optimizing a Machine Learning model in production — from API creation to data drift detection, using the real Home Credit Default Risk dataset from Kaggle

Think about it: you’ve built a great model in a Jupyter notebook. It achieves 0.85 AUC. Everyone’s happy. But then what? The model just sits in your notebook. How does the loan officer at the bank actually use it to make decisions? How do you know it’s still accurate 6 months from now? What happens when it breaks at 3 AM on a Friday?

MLOps (Machine Learning Operations) solves. It’s the set of practices that takes a model from “it works on my laptop” to “it runs reliably in production, 24/7, and we know when something goes wrong.”

What we’ll build together:

1. A prediction API using FastAPI that serves a credit scoring model trained on real Kaggle data
2. Automated tests to make sure our API doesn’t break
3. A Docker container to package everything for deployment
4. A CI/CD pipeline with GitHub Actions to automate testing and deployment
5. A data drift analysis to monitor model health over time
6. Performance optimizations to speed up inference

### Create the project directory skeleton

Run on terminal (or you can do it manually) the following comands:

```bash
mkdir credit-scoring-mlops
cd credit-scoring-mlops
mkdir -p app model tests notebooks monitoring .github/workflows

### Initialize Git

Turning this folder into a Git repository. Git tracks every change you make, creating a history you can go back to if something breaks

Run the following comnd on terminal nd follow the steps to sync with remote GitHub account and make the first commit

```bash
git init
```
### First commit (.gitignore)

```bash
git add .gitignore
git commit -m "Initial commit: project structure and .gitignore"



### Load required libs

In [1]:
import pandas as pd
import numpy as np

# need to load utils
import sys
import os

sys.path.append(os.path.abspath(".."))

### Load data

In [2]:
df = pd.read_csv('../data/application_train.csv')

print(f"Shape: {df.shape}")
print(f"Columns: {df.shape[1]}")
print(f"Rows: {df.shape[0]:,}")

Shape: (307511, 122)
Columns: 122
Rows: 307,511


##### Target column

The target column TARGET is binary:

* 0 = the applicant repaid the loan successfully
* 1 = the applicant had payment difficulties (default)

In [3]:
print(f"\nTarget distribution:")
print(df['TARGET'].value_counts(normalize=True))


Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


### Select features
Extracting only the columns we need from the full 122-column dataframe. 

Working with 15 chosen features is more manageable than 122, and for our API, each feature will become a field that users need to provide. 
* 15 fields is reasonable; 122 is not.

###  Fix the DAYS_EMPLOYED anomaly

* 1,000 years of employment — obviously a placeholder, not a real value. 

If we leave it, it would massively skew our model because the scaler would treat it as a real data point, compressing all actual employment durations into a tiny range.

In [4]:
selected_features = [
 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH',
 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
 'CNT_CHILDREN', 'NAME_EDUCATION_TYPE',
]

X = df[selected_features].copy()
y = df['TARGET'].copy()

In [5]:
# How many rows have this anomalous value?
print(f"Anomalous DAYS_EMPLOYED values: {(df['DAYS_EMPLOYED'] == 365243).sum():,}")

Anomalous DAYS_EMPLOYED values: 55,374


In [6]:
X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

### Convert days to years

In [7]:
# DAYS_BIRTH is negative: -14585 means the person was born 14585 days before the application
# Dividing by -365.25 converts to positive years (365.25 accounts for leap years)
X['AGE_YEARS'] = (-X['DAYS_BIRTH'] / 365.25).round(1)

# DAYS_EMPLOYED works the same way
X['YEARS_EMPLOYED'] = (-X['DAYS_EMPLOYED'] / 365.25).round(1)

# DAYS_ID_PUBLISH: years since the ID document was issued
X['YEARS_ID_PUBLISH'] = (-X['DAYS_ID_PUBLISH'] / 365.25).round(1)

In [8]:
X = X.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH'])

### Encode categorical variables

In [9]:
X['CODE_GENDER'] = X['CODE_GENDER'].map({'M': 0, 'F': 1}).fillna(0).astype(int)
X['FLAG_OWN_CAR'] = X['FLAG_OWN_CAR'].map({'N': 0, 'Y': 1}).astype(int)
X['FLAG_OWN_REALTY'] = X['FLAG_OWN_REALTY'].map({'N': 0, 'Y': 1}).astype(int)

In [10]:
education_map = {
 'Lower secondary': 0,
 'Secondary / secondary special': 1,
 'Incomplete higher': 2,
 'Higher education': 3,
 'Academic degree': 4,
}
X['EDUCATION_LEVEL'] = X['NAME_EDUCATION_TYPE'].map(education_map).fillna(1).astype(int)
X = X.drop(columns=['NAME_EDUCATION_TYPE'])

###  Handle missing values

In [11]:
print("Missing values before filling:")
missing = X.isnull().sum()
print(missing[missing > 0])

Missing values before filling:
EXT_SOURCE_1       173378
EXT_SOURCE_2          660
EXT_SOURCE_3        60965
AMT_ANNUITY            12
AMT_GOODS_PRICE       278
YEARS_EMPLOYED      55374
dtype: int64


In [12]:
X = X.fillna(X.median())

print("\nMissing values after filling:", X.isnull().sum().sum()) # Should be 0


Missing values after filling: 0


### Feature engineering

Creating new features by combining existing ones. 

In [13]:
# Raatio variables

# Credit-to-income ratio: how many years of income does the loan represent?
# A ratio of 5 means the loan is 5x the annual income — that's a heavy burden
X['CREDIT_INCOME_RATIO'] = X['AMT_CREDIT'] / (X['AMT_INCOME_TOTAL'] + 1)

In [14]:
# Annuity-to-income ratio: what fraction of income goes to monthly payments?
# A ratio of 0.3 means 30% of income goes to loan payments — that's stressful
X['ANNUITY_INCOME_RATIO'] = X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)

# Credit-to-goods ratio: how much of the goods price is financed?
# A ratio close to 1 means the person is financing nearly 100% of the purchase
# (no down payment), which is a risk signal
X['CREDIT_GOODS_RATIO'] = X['AMT_CREDIT'] / (X['AMT_GOODS_PRICE'] + 1)

### Final feature set

In [15]:
print(f"Final feature set: {X.shape[1]} features, {X.shape[0]:,} rows")
print(f"Features: {list(X.columns)}")
print(f"Any remaining NaN: {X.isnull().any().any()}")

Final feature set: 18 features, 307,511 rows
Features: ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AGE_YEARS', 'YEARS_EMPLOYED', 'YEARS_ID_PUBLISH', 'EDUCATION_LEVEL', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_GOODS_RATIO']
Any remaining NaN: False


### Make all steps as a function - Remember to add logs

In [16]:
import sys
!{sys.executable} -m pip install loguru


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
from loguru import logger
logger.info("Loguru working 🚀")

2026-04-01 08:46:21.405 | INFO     | __main__:<module>:2 - Loguru working 🚀


In [ ]:
# prepare_data.py

import pandas as pd
import numpy as np
from utils.logger import logger
import sys


def load_and_prepare_data(filepath='../data/application_train.csv'):
    logger.info("Prepare data")
    logger.info("Loading dataset from {}", filepath)

    df = pd.read_csv(filepath)
    
    selected_features = [
        'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
        'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH',
        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
        'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY',
        'CNT_CHILDREN', 'NAME_EDUCATION_TYPE',
    ]

    X = df[selected_features].copy()
    y = df['TARGET'].copy()

    X['DAYS_EMPLOYED'] = X['DAYS_EMPLOYED'].replace(365243, np.nan)

    X['AGE_YEARS'] = (-X['DAYS_BIRTH'] / 365.25).round(1)
    X['YEARS_EMPLOYED'] = (-X['DAYS_EMPLOYED'] / 365.25).round(1)
    X['YEARS_ID_PUBLISH'] = (-X['DAYS_ID_PUBLISH'] / 365.25).round(1)
    X = X.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_ID_PUBLISH'])

    X['CODE_GENDER'] = X['CODE_GENDER'].map({'M': 0, 'F': 1}).fillna(0).astype(int)
    X['FLAG_OWN_CAR'] = X['FLAG_OWN_CAR'].map({'N': 0, 'Y': 1}).astype(int)
    X['FLAG_OWN_REALTY'] = X['FLAG_OWN_REALTY'].map({'N': 0, 'Y': 1}).astype(int)

    education_map = {
        'Lower secondary': 0, 'Secondary / secondary special': 1,
        'Incomplete higher': 2, 'Higher education': 3, 'Academic degree': 4,
    }

    X['EDUCATION_LEVEL'] = X['NAME_EDUCATION_TYPE'].map(education_map).fillna(1).astype(int)
    X = X.drop(columns=['NAME_EDUCATION_TYPE'])

    X = X.fillna(X.median())

    X['CREDIT_INCOME_RATIO'] = X['AMT_CREDIT'] / (X['AMT_INCOME_TOTAL'] + 1)def model.
    X['ANNUITY_INCOME_RATIO'] = X['AMT_ANNUITY'] / (X['AMT_INCOME_TOTAL'] + 1)
    X['CREDIT_GOODS_RATIO'] = X['AMT_CREDIT'] / (X['AMT_GOODS_PRICE'] + 1)
    
    logger.info(f"Data prepared: X shape={X.shape}, y shape={y.shape}")
    logger.info("Default rate: {:.2%}", y.mean())
    return X, y

if __name__ == '__main__':
    X, y = load_and_prepare_data()
    
    # print(f"Features: {X.shape}, Target: {y.shape}")
    # print(f"Default rate: {y.mean():.2%}")

2026-04-01 08:46:21.455 | INFO     | __main__:load_and_prepare_data:10 - Prepare data
2026-04-01 08:46:21.458 | INFO     | __main__:load_and_prepare_data:11 - Loading dataset from ../data/application_train.csv


2026-04-01 08:46:25.337 | INFO     | __main__:load_and_prepare_data:51 - Data prepared: X shape=(307511, 18), y shape=(307511,)
2026-04-01 08:46:25.339 | INFO     | __main__:load_and_prepare_data:52 - Default rate: 8.07%


## Part B - Training the Credit Scoring Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score


X_train, X_test, y_train, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set: {X_test.shape[0]:,} rows")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate: {y_test.mean():.2%}")

Training set: 246,008 rows
Test set: 61,503 rows
Train default rate: 8.07%
Test default rate: 8.07%


### Build a scikit-learn Pipeline

Pipeline, deployment looks like this:

1. Load the scaler
2. Transform the data with the scaler
3. Load the model
4. Feed the transformed data to the model

* The classifier builds 200 decision trees, each trying to better predict defaults.

In [20]:
pipeline = Pipeline([
 ('scaler', StandardScaler()),
 ('classifier', GradientBoostingClassifier(
 n_estimators=200,
 max_depth=4,
 learning_rate=0.1,
 subsample=0.8,
 random_state=42,
 ))
])

pipeline.fit(X_train, y_train)

,steps,"[('scaler', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,loss,'log_loss'
,learning_rate,0.1
,n_estimators,200
,subsample,0.8


### Evaluate

* Measuring how well the model performs on data it has never seen. 

* The training score tells you how well the model memorized the training data> 

* The test score tells you how well it will actually perform in production.

In [ ]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96     56538
           1       0.51      0.03      0.05      4965

    accuracy                           0.92     61503
   macro avg       0.72      0.51      0.50     61503
weighted avg       0.89      0.92      0.88     61503

ROC AUC Score: 0.7554


### Takeaways:

* Due of the imbalance, a model that always predicts “no default” would have 92% accuracy — sounds great, right? 

    * But it would be completely useless because it never identifies actual defaulters. 

    * ROC AUC measures how well the model ranks defaulters above non-defaulters, regardless of the threshold.

### Save data and pkl file

In [23]:
import joblib
import os

BASE_DIR = os.path.abspath("..")  # go up from notebooks/

MODEL_DIR = os.path.join(BASE_DIR, "model")
DATA_DIR = os.path.join(BASE_DIR, "data")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

joblib.dump(pipeline, os.path.join(MODEL_DIR, "credit_model.pkl"))

X_train.to_csv(os.path.join(DATA_DIR, "reference_data.csv"), index=False)
X_test.to_csv(os.path.join(DATA_DIR, "test_data.csv"), index=False)

joblib.dump(list(X_train.columns), os.path.join(MODEL_DIR, "feature_columns.pkl"))

['/home/max/projects/credit-scoring-mlops/model/feature_columns.pkl']

In [ ]:
# trin_model.py

from utils.logger import logger
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score


def model_gbm(df_X, target):

    logger.info("Starting GBM model training...")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        df_X, target, test_size=0.2, random_state=42, stratify=target
    )

    # Pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', GradientBoostingClassifier(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42,
        ))
    ])

    # Train
    pipeline.fit(X_train, y_train)

    logger.info("GBM Model training completed")

    # Evaluate
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    auc = roc_auc_score(y_test, y_proba)

    logger.info("Model AUC: {:.4f}", auc)
    logger.debug("\n{}", classification_report(y_test, y_pred))

    # Save model
    BASE_DIR = os.path.abspath("..")
    MODEL_DIR = os.path.join(BASE_DIR, "model")

    os.makedirs(MODEL_DIR, exist_ok=True)

    model_path = os.path.join(MODEL_DIR, "credit_model.pkl")
    joblib.dump(pipeline, model_path)
    logger.success("Model successfully saved at {}", model_path)

    # Save feature list
    feature_columns = list(X_train.columns)
    feature_path = os.path.join(MODEL_DIR, "feature_columns.pkl")
    joblib.dump(feature_columns, feature_path)
    logger.success("Feature list saved at {}", feature_path)

    # Save feature importance
    feature_importance = pipeline.named_steps['classifier'].feature_importances_
    logger.info("Feature importance computed")

    return pipeline, auc, feature_importance, feature_columns

if __name__ == '__main__':
    pipeline, auc, feature_importance, feature_columns = model_gbm(X, y)

2026-04-01 09:35:45.454 | INFO     | __main__:model_gbm:16 - Starting GBM model training...
2026-04-01 09:38:25.808 | INFO     | __main__:model_gbm:38 - GBM Model training completed
2026-04-01 09:38:26.186 | INFO     | __main__:model_gbm:46 - Model AUC: 0.7554
2026-04-01 09:38:26.216 | SUCCESS  | __main__:model_gbm:57 - Model successfully saved at /home/max/projects/credit-scoring-mlops/model/credit_model.pkl
2026-04-01 09:38:26.218 | SUCCESS  | __main__:model_gbm:63 - Feature list saved at /home/max/projects/credit-scoring-mlops/model/feature_columns.pkl
2026-04-01 09:38:26.220 | INFO     | __main__:model_gbm:67 - Feature importance computed
